In [1]:
%load_ext autoreload
%autoreload 2
# conda activate fishyrl
# tensorboard --logdir ./examples/runs

TODO Ordered List
- Add configuration file loading
- Move all functions to a separate file and make notebook into script
- Make tensorboard logging optional
- Implement loading
- Add video logging to tensorboard, then add to README
- Add timers
- Refactor initialization
- Test or revise discretization and continuous actions (don't forget about objective in `train`)
- Add mixed precision training, or at least fp16
- Implement https://rlgym.org/
- Implement distributed
- Add missing tests
- Vectorize discrete and discretized actions, as well as buffer storage and sampling

In [ ]:
import math

import fishyrl as frl

In [ ]:
# Tested
# CartPole-v1: Begins converging to optimal after 10 minutes w/ 512 EOD, 2 NB, 512 DD, 32 SD, 512 DD, and sequence length 10
# MountainCar-v0: Doesn't get an initial reward to work from with same params as CartPole-v1
# BipedalWalker-v3: Learns to stand
# https://github.com/Eclectic-Sheep/sheeprl/issues/229

In [ ]:
# Load config
cfg = frl.utilities.load_config('../examples/CartPole.yaml')

# Load environment and actions
envs, actions = frl.dreamer.get_environments_and_actions(cfg=cfg)

# Construct models
world_model, agent_model, utility_modules = frl.dreamer.construct_models(
    env_obs_dim=math.prod(envs.observation_space.shape[1:]),  # TODO: Revise for CNN
    env_actions=actions,
    device='cuda',
    cfg=cfg,
)

# Tensorboard writer
# folder_name = f'{env_name}_{time.strftime("%Y-%m-%d_%H-%M-%S")}'
# folder_name = f'./runs/{folder_name}'
# writer = torch.utils.tensorboard.SummaryWriter(folder_name)

In [ ]:
# environment_steps = 10**6  # Guessing
# start_train_step = 1024  # TODO: In the case of loading a checkpoint, don't randomly sample, but instead use the model until the starting training step
# batch_size = 16  # Might need to refine, based on paper
# sequence_length = 64  # Might need to refine, based on paper
# imagination_horizon = 15
# gamma = .997
# lmbda = .95
# tau = .02  # Target critic update rate

# def train(batch: dict[str, torch.Tensor], environment_step: int) -> None:
#     """Single-step training function."""
#     # Infer batch size and sequence length (batch, sequence_length, ...)
#     batch_size, sequence_length = batch['obs'].shape[:2]

#     # Embed observations
#     embedded_obs = encoder_model(batch['obs'])

#     # Initialize storage
#     hidden_states = []
#     priors = []
#     priors_logits = []
#     posteriors = []
#     posteriors_logits = []

#     # SheepRL does this, but creates an issue with autograd and doesn't appear to be faster
#     # hidden_states = torch.empty((batch_size, sequence_length, DETERMINISTIC_DIM))
#     # priors = torch.empty((batch_size, sequence_length, STOCHASTIC_DIM * CATEGORICAL_BINS))
#     # priors_logits = torch.empty((batch_size, sequence_length, STOCHASTIC_DIM * CATEGORICAL_BINS))
#     # posteriors = torch.empty((batch_size, sequence_length, STOCHASTIC_DIM * CATEGORICAL_BINS))
#     # posteriors_logits = torch.empty((batch_size, sequence_length, STOCHASTIC_DIM * CATEGORICAL_BINS))

#     # Compute model outputs for each time step, starting with initial recurrent state and posteriors
#     for i in range(sequence_length):
#         # Run through recurrent model
#         ret = rssm_model(
#             batch['actions'][:, i-1] if i > 0 else None,  # Use the action from the previous step, like in the environment loop
#             posteriors[i-1] if i > 0 else None,
#             hidden_states[i-1] if i > 0 else None,
#             embedded_obs[:, i],
#             batch['terminations'][:, i-1] | batch['truncations'][:, i-1] if i > 0 else None,  # Get initializations using result of previous step
#             batch_dim=batch_size,
#         )
#         hidden_states.append(ret['hidden_state'])
#         priors.append(ret['prior'])
#         priors_logits.append(ret['prior_logits'])
#         posteriors.append(ret['posterior'])
#         posteriors_logits.append(ret['posterior_logits'])

#         # SheepRL assignment approach
#         # TODO: Figure out why this doesn't throw an error in SheepRL (It does right now)
#         # hidden_states[:, i] = ret['hidden_state']
#         # priors[:, i] = ret['prior']
#         # priors_logits[:, i] = ret['prior_logits']
#         # posteriors[:, i] = ret['posterior']
#         # posteriors_logits[:, i] = ret['posterior_logits']

#     # Concatenate returned tensors
#     hidden_states = torch.stack(hidden_states, dim=1)
#     priors = torch.stack(priors, dim=1)
#     priors_logits = torch.stack(priors_logits, dim=1)
#     posteriors = torch.stack(posteriors, dim=1)
#     posteriors_logits = torch.stack(posteriors_logits, dim=1)

#     # Compute predicted observations, rewards, and continues
#     pred_obs = decoder_model(torch.cat((posteriors, hidden_states), dim=-1))
#     pred_rewards = reward_model(torch.cat((posteriors, hidden_states), dim=-1))
#     pred_continues = continue_model(torch.cat((posteriors, hidden_states), dim=-1))

#     # Compute MSE loss for reconstructed observations (po)
#     # observation_loss = frl.losses.mse_loss(pred_obs, XXX, dims=2)  # CNN
#     observation_loss = frl.losses.mse_loss(pred_obs, frl.distributions.symlog(batch['obs'])).clip(min=1e-8)  # MLP

#     # Compute reward loss (pr)
#     dist_pred_rewards = frl.distributions.TwoHot(pred_rewards)  # NOTE: Default reward range is -20, 20
#     reward_loss = - dist_pred_rewards.log_prob(batch['rewards'])  # TODO: Verify that this is the correct timestep, same with continues

#     # Compute continue loss (pc)
#     dist_pred_continues = torch.distributions.Bernoulli(logits=pred_continues.squeeze(-1))
#     continue_targets = 1 - 1. * batch['terminations']
#     continue_loss = - dist_pred_continues.log_prob(continue_targets)

#     # Reshape priors and posteriors
#     posteriors_logits = posteriors_logits.reshape([*posteriors_logits.shape[:-1], -1, CATEGORICAL_BINS])
#     priors_logits = priors_logits.reshape([*priors_logits.shape[:-1], -1, CATEGORICAL_BINS])

#     # KL Balancing
#     dynamic_loss = torch.distributions.kl_divergence(
#         torch.distributions.Independent(torch.distributions.OneHotCategoricalStraightThrough(logits=posteriors_logits.detach()), 1),
#         torch.distributions.Independent(torch.distributions.OneHotCategoricalStraightThrough(logits=priors_logits), 1))
#     representation_loss = torch.distributions.kl_divergence(
#         torch.distributions.Independent(torch.distributions.OneHotCategoricalStraightThrough(logits=posteriors_logits), 1),
#         torch.distributions.Independent(torch.distributions.OneHotCategoricalStraightThrough(logits=priors_logits.detach()), 1))
#     free_nats = torch.tensor(1.)
#     dynamic_loss = torch.maximum(dynamic_loss, free_nats)  # Free nats
#     representation_loss = torch.maximum(representation_loss, free_nats)  # Free nats
#     kl_loss = .5 * dynamic_loss + .1 * representation_loss

#     # Total loss
#     reconstruction_loss = (1 * kl_loss + observation_loss + reward_loss + 1 * continue_loss).mean()

#     # Log to tensorboard
#     writer.add_scalar('Loss/World', reconstruction_loss.item(), environment_step)
#     writer.add_scalar('World/Observation', observation_loss.mean().item(), environment_step)
#     writer.add_scalar('World/Reward', reward_loss.mean().item(), environment_step)
#     writer.add_scalar('World/Continue', continue_loss.mean().item(), environment_step)
#     writer.add_scalar('World/Dynamic', dynamic_loss.mean().item(), environment_step)
#     writer.add_scalar('World/Representation', representation_loss.mean().item(), environment_step)

#     # Step world model
#     world_optimizer.zero_grad()
#     reconstruction_loss.backward()
#     nn.utils.clip_grad_norm_(world_params, max_norm=1000.)
#     world_optimizer.step()

#     # Behavior learning
#     # imagined_prior = posteriors.detach().reshape(-1, posteriors.shape[-1])
#     # imagined_hidden_state = hidden_states.detach().reshape(-1, hidden_states.shape[-1])
#     imagined_prior, imagined_hidden_state = posteriors.detach(), hidden_states.detach()

#     # Initialize storage
#     imagined_trajectories = [torch.cat((imagined_prior, imagined_hidden_state), dim=-1)]
#     imagined_actions = []

#     # Imagine for `imagination_horizon` steps
#     for _ in range(imagination_horizon):
#         # Compute action based on previous prior + hidden state
#         actions, action_distributions = actor_model(imagined_trajectories[-1].detach())  # This detach was added April 8th, 2026, and after cartpole 19-07

#         # Imagine future prior
#         ret = rssm_model(actions, imagined_prior, imagined_hidden_state)
#         imagined_prior = ret['prior']
#         imagined_hidden_state = ret['hidden_state']

#         # Record
#         imagined_actions.append(actions)
#         imagined_trajectories.append(torch.cat((imagined_prior, imagined_hidden_state), dim=-1))

#     # Stack to (batch, sequence, horizon(-1), features)
#     imagined_trajectories = torch.stack(imagined_trajectories, dim=2)
#     imagined_actions = torch.stack(imagined_actions, dim=2)

#     # Compute predicted rewards, values, and continues
#     pred_rewards = reward_model(imagined_trajectories)
#     pred_rewards = frl.distributions.TwoHot(pred_rewards).mean
#     pred_values = critic_model(imagined_trajectories)
#     pred_values = frl.distributions.TwoHot(pred_values).mean
#     pred_continues = continue_model(imagined_trajectories[:, :, 1:])
#     pred_continues = torch.distributions.Bernoulli(logits=pred_continues.squeeze(-1)).mode
#     # continues = 1 - 1. * (batch['terminations'] | batch['truncations'])  # Use actual continues, including truncations
#     pred_continues = torch.cat((continue_targets.unsqueeze(-1), pred_continues), dim=-1)  # Use actual continues where possible

#     # Compute lambda values
#     # NOTE: These are not advantages
#     # https://github.com/Eclectic-Sheep/sheeprl/blob/33b636681fd8b5340b284f2528db8821ab8dcd0b/sheeprl/algos/dreamer_v3/utils.py#L66
#     # https://github.com/danijar/dreamerv3/blob/b65cf81a6fb13625af8722127459283f899a35d9/dreamerv3/agent.py#L482
#     lambda_values = [pred_values[:, :, -1]]  # Add bootstrapping value
#     interm = pred_rewards[:, :, 1:] + pred_continues[:, :, 1:] * gamma * pred_values[:, :, 1:] * (1 - lmbda)
#     for i in range(pred_rewards[:, :, 1:].shape[2] - 1, -1, -1):
#         lambda_values.append( interm[:, :, i] + pred_continues[:, :, 1:][:, :, i] * gamma * lmbda * lambda_values[-1] )
#     lambda_values = torch.stack(lambda_values[:0:-1], dim=2)

#     # Using previous values as baseline, compute advantage
#     baseline = pred_values[:, :, :-1]
#     norm_low, norm_range = lambda_normalizer(lambda_values)
#     normalized_lambda_values = (lambda_values - norm_low) / norm_range
#     normalized_baseline = (baseline - norm_low) / norm_range
#     advantage = normalized_lambda_values - normalized_baseline

#     # Compute discounts based on horizon, and disregard after non-continues
#     horizon_discount = torch.cumprod(pred_continues[:, :, :-1] * gamma, dim=2) / gamma
#     horizon_discount = horizon_discount.detach()

#     # Compute objective by summing action log probabilities
#     # TODO: Maybe this could be computed earlier? We could do it in the imagination loop, but it would propagate gradients throughout
#     #       the horizon on the side of the distribution, which is not in the implementation.
#     _, action_distributions = actor_model(imagined_trajectories[:, :, :-1].detach())
#     objective = 0
#     for dist, action in zip(action_distributions, imagined_actions.split(MODEL_ACTION_OUTPUT_DIMS, dim=-1)):
#         objective += dist.log_prob(action.detach())
#     # objective = advantage  # Continuous
#     objective = advantage.detach() * objective
#     # TODO TODO TODO BOOTSTRAP IN THE TRUNCATION CASE

#     # Compute entropy
#     entropy = 3e-4 * sum([dist.entropy() for dist in action_distributions])

#     # Compute actor loss
#     actor_loss = - horizon_discount.detach() * (objective + entropy)
#     actor_loss = actor_loss.mean()

#     # Log to tensorboard
#     writer.add_scalar('Loss/Actor', actor_loss.item(), environment_step)
#     writer.add_scalar('Actor/Objective', - objective.mean().item(), environment_step)
#     writer.add_scalar('Actor/Entropy', - entropy.mean().item(), environment_step)

#     # Step actor model
#     actor_optimizer.zero_grad()
#     actor_loss.backward()
#     nn.utils.clip_grad_norm_(actor_model.parameters(), max_norm=100.)
#     actor_optimizer.step()

#     # Compute predicted critic and target critic values
#     # TODO: Could do this earlier, but the stored gradients may complicate backpropagation, time-wise.
#     pred_values = critic_model(imagined_trajectories[:, :, :-1].detach())
#     dist_pred_values = frl.distributions.TwoHot(pred_values)
#     pred_target_values = frl.distributions.TwoHot(target_critic_model(imagined_trajectories[:, :, :-1].detach())).mean

#     # Compute critic loss
#     target_loss = - dist_pred_values.log_prob(lambda_values.detach())
#     divergence_loss = - dist_pred_values.log_prob(pred_target_values.detach())  # Don't stray too far from target critic values
#     critic_loss = target_loss + divergence_loss
#     critic_loss = horizon_discount.detach() * critic_loss  # Discount based on horizon
#     critic_loss = critic_loss.mean()

#     # Log to tensorboard
#     writer.add_scalar('Loss/Critic', critic_loss.item(), environment_step)
#     writer.add_scalar('Critic/Target', target_loss.mean().item(), environment_step)
#     writer.add_scalar('Critic/Divergence', divergence_loss.mean().item(), environment_step)

#     # Step critic model
#     critic_optimizer.zero_grad()
#     critic_loss.backward()
#     nn.utils.clip_grad_norm_(critic_model.parameters(), max_norm=100.)
#     critic_optimizer.step()

#     # Reset gradients
#     world_optimizer.zero_grad(set_to_none=True)
#     actor_optimizer.zero_grad(set_to_none=True)
#     critic_optimizer.zero_grad(set_to_none=True)


# def main() -> None:
#     """Main training loop."""
#     # Initialize variables
#     actions = posteriors = hidden_states = initializations = None
#     rewards, terminations, truncations = np.zeros(num_envs), np.zeros(num_envs, dtype=bool), np.zeros(num_envs, dtype=bool)
#     # pred_rewards = pred_continues = 0

#     # Tracking
#     cumulative_rewards = np.zeros(num_envs)

#     # Loop for specified number of iterations
#     obs, info = envs.reset(seed=42)
#     for environment_step in range(0, environment_steps, num_envs):
#         ## Compute an environment step
#         if environment_step >= start_train_step:
#             with torch.no_grad():
#                 # Embed observation
#                 embedded_obs = encoder_model(torch.from_numpy(obs).to(DEVICE))  # TODO: GPU

#                 # Compute hidden states
#                 ret = rssm_model(
#                     actions,
#                     posteriors,
#                     hidden_states,
#                     embedded_obs,
#                     initializations,
#                     batch_dim=envs.num_envs,
#                 )
#                 hidden_states = ret['hidden_state']
#                 # priors = ret['prior']
#                 # prior_logits = ret['prior_logits']
#                 posteriors = ret['posterior']
#                 # posterior_logits = ret['posterior_logits']

#                 # Decode observation
#                 # pred_obs = decoder_model(torch.cat((posteriors, hidden_states), dim=-1))

#                 # Predict reward and continue
#                 # TODO: Double-check, but compute reward for the previous action
#                 # pred_rewards = reward_model(torch.cat((posteriors, hidden_states), dim=-1))
#                 # pred_continues = continue_model(torch.cat((posteriors, hidden_states), dim=-1))

#                 # Compute actions
#                 actions, action_distributions = actor_model(torch.cat((posteriors, hidden_states), dim=-1))

#                 # Sample final actions
#                 env_actions = frl.actions.simplify_actions(actions, MODEL_ACTIONS).detach().cpu().numpy()
#                 if env_actions.shape[-1] == 1: env_actions = env_actions.squeeze(-1)  # Remove when using >1 action spaces
#         # Compute action randomly if not training yet
#         else:
#             sampled_actions = envs.action_space.sample().reshape(num_envs, -1)
#             actions = frl.actions.construct_actions(torch.tensor(sampled_actions, dtype=torch.get_default_dtype()), MODEL_ACTIONS)
#             env_actions = sampled_actions.squeeze(-1) if sampled_actions.shape[-1] == 1 else sampled_actions  # Remove when using >1 action spaces

#         # Record to buffer
#         buffer.add({
#             # Environment-related experiences
#             'obs': obs,
#             'rewards': rewards,
#             'terminations': terminations,
#             'truncations': truncations,
#             # Predicted environment experiences
#             # 'pred_rewards': pred_rewards.detach().cpu().numpy(),
#             # 'pred_continues': pred_continues.detach().cpu().numpy(),
#             # 'pred_obs': pred_obs.detach().cpu().numpy(),
#             # RSSM experience
#             # 'hidden_states': hidden_states.detach().cpu().numpy(),
#             # 'priors': priors.detach().cpu().numpy(),
#             # 'prior_logits': prior_logits.detach().cpu().numpy(),
#             # 'posteriors': posteriors.detach().cpu().numpy(),
#             # 'posterior_logits': posterior_logits.detach().cpu().numpy(),
#             # Actions
#             'actions': actions.detach().cpu().numpy(),
#             # 'action_distributions': action_distributions,
#         })

#         # Step environment
#         obs, rewards, terminations, truncations, infos = envs.step(env_actions)
#         initializations = torch.tensor(terminations | truncations, dtype=torch.bool, device=DEVICE)

#         # Progress indicator
#         print(num_envs * '.', end='')
#         if np.any(terminations | truncations): print()

#         # Iterate and print rewards if done
#         cumulative_rewards += rewards
#         for i in range(num_envs):
#             if terminations[i]:
#                 print(f'({environment_step}) Environment {i + 1} terminated with cumulative reward: {cumulative_rewards[i]}')
#                 writer.add_scalar('Reward/Episode', cumulative_rewards[i], environment_step)
#                 cumulative_rewards[i] = 0
#             elif truncations[i]:
#                 print(f'({environment_step}) Environment {i + 1} truncated with cumulative reward: {cumulative_rewards[i]}')
#                 writer.add_scalar('Reward/Episode', cumulative_rewards[i], environment_step)
#                 cumulative_rewards[i] = 0

#         ## Train
#         if environment_step >= start_train_step:
#             gradient_steps = ratio(environment_step - start_train_step)
#             for _ in range(gradient_steps):
#                 # Sample batch of experiences from buffer and train
#                 batch = buffer.sample(batch_size, sequence_length=sequence_length)
#                 batch = frl.buffers.convert_samples_to_tensors(batch, device=DEVICE)

#                 # Iterate target critic model towards critic model
#                 update_tau = tau if gradient_steps > 0 else 1.
#                 for target_param, param in zip(target_critic_model.parameters(), critic_model.parameters()):
#                     target_param.data.copy_(update_tau * param.data + (1 - update_tau) * target_param.data)

#                 # Run training step
#                 train(batch, environment_step)

In [ ]:
# main()

....................................................
(48) Environment 4 terminated with cumulative reward: 13.0
........
(56) Environment 2 terminated with cumulative reward: 15.0
................
(72) Environment 1 terminated with cumulative reward: 19.0
........................................................
(128) Environment 1 terminated with cumulative reward: 13.0
....
(132) Environment 4 terminated with cumulative reward: 20.0
....................
(152) Environment 3 terminated with cumulative reward: 39.0
........
(160) Environment 2 terminated with cumulative reward: 25.0
................
(176) Environment 1 terminated with cumulative reward: 11.0
....
(180) Environment 4 terminated with cumulative reward: 11.0
....................................................................
(248) Environment 1 terminated with cumulative reward: 17.0
(248) Environment 4 terminated with cumulative reward: 16.0
........................................
(288) Environment 4 terminated with cumu

KeyboardInterrupt: 

In [ ]:
# # Save models and optimizers
# torch.save({
#     'encoder_model': encoder_model.state_dict(),
#     'decoder_model': decoder_model.state_dict(),
#     'rssm_model': rssm_model.state_dict(),
#     'reward_model': reward_model.state_dict(),
#     'continue_model': continue_model.state_dict(),
#     'actor_model': actor_model.state_dict(),
#     'critic_model': critic_model.state_dict(),
#     'target_critic_model': target_critic_model.state_dict(),
#     'buffer': buffer.state_dict(),
#     'lambda_normalizer': lambda_normalizer.state_dict(),
#     'ratio': ratio.state_dict(),
#     'world_optimizer': world_optimizer.state_dict(),
#     'actor_optimizer': actor_optimizer.state_dict(),
#     'critic_optimizer': critic_optimizer.state_dict(),
# }, f'{folder_name}/model.weights')

# # Close writer
# writer.close()

In [ ]:
# {k: v.shape for k, v in buffer.sample(batch_size, sequence_length=sequence_length).items()}

{'obs': (16, 64, 24),
 'rewards': (16, 64),
 'terminations': (16, 64),
 'truncations': (16, 64),
 'actions': (16, 64, 128)}